# Notebook 01: Package Evaluation for Pain EEG Classification

**Dataset**: Zhao et al. 2025 -- OpenNeuro ds005284 (678-subject Biosemi EEG, pain vs no-pain)  
**Target electrodes**: C3, Cz, C4 (Primary Somatosensory Cortex, S1)  
**Goal**: Identify which packages from the candidate list are applicable, installable, and useful for building a multiband pain/no-pain classifier.

Each package is scored on four axes:
- **Data I/O**: Can it load or convert the Biosemi BIDS data?
- **Preprocessing**: Does it contribute filtering, re-referencing, artifact rejection, or epoching?
- **Feature extraction**: Does it contribute spectral, temporal, connectivity, or source-level features?
- **Classification support**: Does it offer ML utilities or variability-removal tools?

Score: 0 (no contribution) to 3 (primary/essential).

In [ ]:
# Install all candidate packages.
# Run this cell once. Some packages install cleanly from PyPI; others require git.

import subprocess, sys

def pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *args],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"FAILED {args}: {result.stderr[:300]}")
    else:
        print(f"OK     {args[0]}")

# Core scientific stack
pip("numpy", "scipy", "pandas", "matplotlib", "seaborn", "scikit-learn")

# Primary EEG processing
pip("mne", "mne-bids")

# Feature extraction
pip("antropy")                                  # entropy / fractal / Hjorth

# Neurophysiology data I/O
pip("neo")                                      # python-neo

# Source imaging (MNE extension)
pip("invertmeeg")                               # 96-solver source inversion

# BEhavioral STate analysis
pip("best-toolbox")

# EEGExtract -- no PyPI package; install deps then fetch script
pip("PyWavelets", "spectrum")                   # EEGExtract dependencies

# eegpipe (git-only)
pip("git+https://github.com/mattpontifex/eegpipe-python.git")

# eeg-expy
pip("eegnb")                                    # eeg-expy package name on PyPI

# eegprep (sccn) -- no PyPI release, use pip+git
# pip("git+https://github.com/sccn/eegprep.git")  # Uncomment if needed

# OpenNeuro downloader
pip("openneuro-py", "tqdm", "requests")

print("\nAll installations attempted.")

In [ ]:
import importlib, textwrap, warnings
warnings.filterwarnings("ignore")

def check_import(module_name, friendly_name=None):
    name = friendly_name or module_name
    try:
        m = importlib.import_module(module_name)
        ver = getattr(m, "__version__", "unknown")
        print(f"  PASS  {name:30s}  v{ver}")
        return True
    except ImportError as e:
        print(f"  FAIL  {name:30s}  ({e})")
        return False

print("=" * 65)
print("Import check")
print("=" * 65)
packages = [
    ("mne",         "mne-tools (MNE)"),
    ("mne_bids",    "mne-bids"),
    ("antropy",     "antropy"),
    ("neo",         "python-neo"),
    ("invertmeeg",  "invertmeeg"),
    ("best",        "best-toolbox"),
    ("eegpipe",     "eegpipe-python"),
    ("eegnb",       "eeg-expy (eegnb)"),
    ("PyWavelets",  "PyWavelets (EEGExtract dep)"),
    ("spectrum",    "spectrum (EEGExtract dep)"),
]
results = {name: check_import(mod, name) for mod, name in packages}
print("=" * 65)

In [ ]:
import pandas as pd

# Structured evaluation table.
# Scores: 0=no contribution, 1=minor, 2=useful, 3=primary

records = [
    {
        "Package": "mne-tools (MNE)",
        "DataIO": 3,
        "Preprocessing": 3,
        "Features": 2,
        "Classification": 1,
        "Install": "pip install mne",
        "Status": "ADOPTED",
        "Notes": "Core pipeline engine. Loads Biosemi BDF via BIDS, bandpass, ICA, epoch extraction, PSD, connectivity (mne-connectivity). Essential."
    },
    {
        "Package": "mne-bids",
        "DataIO": 3,
        "Preprocessing": 1,
        "Features": 0,
        "Classification": 0,
        "Install": "pip install mne-bids",
        "Status": "ADOPTED",
        "Notes": "BIDS path resolution, event reading, participant metadata. Required for OpenNeuro ds005284."
    },
    {
        "Package": "antropy",
        "DataIO": 0,
        "Preprocessing": 0,
        "Features": 3,
        "Classification": 0,
        "Install": "pip install antropy",
        "Status": "ADOPTED",
        "Notes": "Sample entropy, permutation entropy, spectral entropy, Higuchi FD, Hjorth params, Petrosian FD. Fast, NumPy-native. Primary nonlinear feature extractor."
    },
    {
        "Package": "EEGExtract",
        "DataIO": 0,
        "Preprocessing": 0,
        "Features": 3,
        "Classification": 0,
        "Install": "Download EEGExtract.py + pip install -r requirements.txt",
        "Status": "ADOPTED (manual)",
        "Notes": "Band power, MFCC, ARMA, coherence, PLI, cross-correlation, Granger causality, Lyapunov. Complements antropy. No PyPI release; single-file import."
    },
    {
        "Package": "invertmeeg",
        "DataIO": 0,
        "Preprocessing": 0,
        "Features": 2,
        "Classification": 0,
        "Install": "pip install invertmeeg",
        "Status": "PARTIAL",
        "Notes": "96 inverse solvers (beamforming, min-norm, sparse, Bayesian). Useful for source-level variability removal but requires a forward model (BEM + MRI). Adds S1 spatial specificity."
    },
    {
        "Package": "python-neo",
        "DataIO": 2,
        "Preprocessing": 0,
        "Features": 0,
        "Classification": 0,
        "Install": "pip install neo",
        "Status": "OPTIONAL",
        "Notes": "Reads many raw neurophysiology formats. Not needed if MNE-BIDS handles BDF loading directly. Useful for mixed-format archives or cross-lab data."
    },
    {
        "Package": "best-toolbox",
        "DataIO": 0,
        "Preprocessing": 1,
        "Features": 2,
        "Classification": 2,
        "Install": "pip install best-toolbox",
        "Status": "PARTIAL",
        "Notes": "EEG state-analysis (originally sleep/seizure). ERP utilities and feature/classification submodules are reusable. Classification wrappers reduce boilerplate."
    },
    {
        "Package": "eegpipe-python",
        "DataIO": 1,
        "Preprocessing": 2,
        "Features": 0,
        "Classification": 0,
        "Install": "pip install git+https://github.com/mattpontifex/eegpipe-python.git",
        "Status": "PARTIAL",
        "Notes": "Baseline correction, epoch-to-continuous, voltage threshold rejection, simple z-wave, topographic headplot. Useful for epoch QC and quick topographic checks on C3/Cz/C4."
    },
    {
        "Package": "eeg-expy (eegnb)",
        "DataIO": 1,
        "Preprocessing": 1,
        "Features": 1,
        "Classification": 1,
        "Install": "pip install eegnb",
        "Status": "OPTIONAL",
        "Notes": "Designed for online EEG experiment delivery and quick offline ERN/P300 analysis. Some overlap with MNE pipeline. Limited added value for the pain classification task."
    },
    {
        "Package": "eegprep (sccn)",
        "DataIO": 0,
        "Preprocessing": 2,
        "Features": 0,
        "Classification": 0,
        "Install": "pip install git+https://github.com/sccn/eegprep.git",
        "Status": "OPTIONAL",
        "Notes": "Python port of EEGLAB prep pipeline (robust re-reference, bridged channels). Useful if reference noise is a concern in Biosemi data. No active PyPI release as of 2025."
    },
]

df = pd.DataFrame(records)
df["TotalScore"] = df[["DataIO", "Preprocessing", "Features", "Classification"]].sum(axis=1)
df = df.sort_values("TotalScore", ascending=False).reset_index(drop=True)
print(df[["Package", "Status", "TotalScore", "DataIO", "Preprocessing", "Features", "Classification"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: grouped bar chart
ax = axes[0]
pkgs = df["Package"].tolist()
dims = ["DataIO", "Preprocessing", "Features", "Classification"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]
x = np.arange(len(pkgs))
width = 0.2
for i, (dim, color) in enumerate(zip(dims, colors)):
    ax.bar(x + i * width, df[dim], width, label=dim, color=color, alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(pkgs, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Score (0-3)")
ax.set_title("Package Scores by Axis")
ax.legend(fontsize=8)
ax.set_ylim(0, 3.5)
ax.grid(axis="y", alpha=0.3)

# Right: radar chart of adopted packages
adopted = df[df["Status"].str.startswith("ADOPTED")]
ax2 = axes[1]
labels = dims
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

ax2 = plt.subplot(1, 2, 2, polar=True)
for _, row in adopted.iterrows():
    values = [row[d] for d in dims] + [row[dims[0]]]
    ax2.plot(angles, values, linewidth=2, label=row["Package"])
    ax2.fill(angles, values, alpha=0.12)
ax2.set_thetagrids(np.degrees(angles[:-1]), labels)
ax2.set_ylim(0, 3)
ax2.set_title("Adopted Packages: Coverage", pad=20)
ax2.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig("package_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: package_evaluation.png")

In [ ]:
# Validate EEGExtract by fetching the single-file script and checking function names.
# This requires internet access. The file is licensed GPL-3.0.

import urllib.request, os, pathlib

eegextract_url = (
    "https://raw.githubusercontent.com/sari-saba-sadiya/EEGExtract/master/EEGExtract.py"
)
eegextract_path = pathlib.Path("EEGExtract.py")

if not eegextract_path.exists():
    try:
        urllib.request.urlretrieve(eegextract_url, eegextract_path)
        print(f"Downloaded EEGExtract.py ({eegextract_path.stat().st_size} bytes)")
    except Exception as e:
        print(f"Download failed: {e}")
else:
    print(f"EEGExtract.py already present ({eegextract_path.stat().st_size} bytes)")

if eegextract_path.exists():
    import ast
    src = eegextract_path.read_text()
    tree = ast.parse(src)
    funcs = [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
    pain_relevant = [
        f for f in funcs if any(
            k in f.lower() for k in ["band", "entropy", "hjorth", "coherence",
                                      "phas", "mi", "granger", "lyap", "mfcc",
                                      "arma", "median", "freq"]
        )
    ]
    print(f"\nTotal functions in EEGExtract: {len(funcs)}")
    print("Pain-relevant functions:")
    for f in sorted(pain_relevant):
        print(f"  {f}")

In [ ]:
# Final recommended pipeline stack

pipeline = """
RECOMMENDED PIPELINE STACK
==========================

Stage 1 -- Data I/O and BIDS navigation
  mne-bids            Read BIDS layout, BidsPaths, participant table, events
  mne (RawBiosemi)    Load BDF files, pick C3/Cz/C4, load sidecar JSONs

Stage 2 -- Preprocessing
  mne                 Notch (50 Hz + harmonics), bandpass (0.5-100 Hz),
                      re-reference (average or linked mastoid),
                      ICA (fastica) for eye/muscle artifact removal,
                      epoch extraction and baseline correction
  eegpipe             Voltage threshold rejection, z-wave epoch QC,
                      topographic headplot for channel QC

Stage 3 -- Feature extraction (per epoch, per electrode: C3, Cz, C4)
  mne (PSD, tfr)      Band power (delta 1-4, theta 4-8, alpha 8-13,
                      beta 13-30, gamma 30-80 Hz) via Welch + multitaper
  antropy             Sample entropy, permutation entropy, spectral entropy,
                      Higuchi FD, Petrosian FD, Hjorth (activity, mobility,
                      complexity), DFA, LZiv complexity
  EEGExtract          MFCC, ARMA coefficients, median frequency,
                      inter-electrode coherence, PLI, cross-correlation lag,
                      Granger causality (C3->Cz, Cz->C4, C3->C4)
  mne-connectivity    Phase-locking value, weighted PLI between C3/Cz/C4

Stage 4 -- Source-level (optional, increases spatial specificity)
  invertmeeg          Beamformer (LCMV) or sLORETA projected to S1 ROI;
                      removes source-level inter-individual variability

Stage 5 -- Variability removal
  scikit-learn        Per-subject z-score (StandardScaler on training folds),
                      RobustScaler (trial-to-trial), PCA whitening
  best-toolbox        State-level normalization utilities where applicable

Stage 6 -- Classification
  scikit-learn        SVM (RBF), LDA, Random Forest, Logistic Regression
                      Leave-One-Subject-Out CV, Stratified-K-Fold
                      ANOVA-F feature selection, permutation importance
"""
print(pipeline)

In [ ]:
# Verify key MNE capabilities relevant to this task

import mne
print(f"MNE version: {mne.__version__}")
print(f"MNE BIDS version: ", end="")
import mne_bids; print(mne_bids.__version__)

# Check antropy
import antropy as ant
import numpy as np

rng = np.random.default_rng(42)
test_signal = rng.standard_normal(1000)  # 1-second synthetic EEG epoch at 1 kHz

print("\nantropy function check on synthetic EEG signal (1000 samples):")
print(f"  sample_entropy:        {ant.sample_entropy(test_signal):.4f}")
print(f"  perm_entropy:          {ant.perm_entropy(test_signal, normalize=True):.4f}")
print(f"  spectral_entropy:      {ant.spectral_entropy(test_signal, sf=1000, method='welch', normalize=True):.4f}")
print(f"  higuchi_fd:            {ant.higuchi_fd(test_signal):.4f}")
print(f"  petrosian_fd:          {ant.petrosian_fd(test_signal):.4f}")
print(f"  detrended_fluctuation: {ant.detrended_fluctuation(test_signal):.4f}")
act, mob, comp = ant.hjorth_params(test_signal)
print(f"  hjorth (act,mob,comp): {act:.4f}, {mob:.4f}, {comp:.4f}")
print(f"  lziv_complexity:       {ant.lziv_complexity(test_signal > 0):.4f}")
print("\nAll antropy functions verified.")

In [ ]:
# Verify python-neo can represent EEG analog signals
import neo
import quantities as pq
import numpy as np

print(f"python-neo version: {neo.__version__}")

# Construct a minimal Neo segment with three EEG channels (C3, Cz, C4)
seg = neo.Segment(name="pain_trial")
for ch_name in ["C3", "Cz", "C4"]:
    sig = neo.AnalogSignal(
        signal=np.random.randn(1024, 1) * 10,
        units="uV",
        sampling_rate=512 * pq.Hz,
        name=ch_name,
        description="Biosemi Biosemi EEG channel"
    )
    seg.analogsignals.append(sig)

print(f"\nNeo segment: {seg.name}")
print(f"  Channels: {[s.name for s in seg.analogsignals]}")
print(f"  Shape per channel: {seg.analogsignals[0].shape}")
print(f"  Sampling rate: {seg.analogsignals[0].sampling_rate}")
print(f"  Units: {seg.analogsignals[0].units}")
print("\npython-neo: can represent Biosemi EEG signal objects. Useful as format bridge; not required in primary pipeline.")

## Summary

| Package | Role in Pipeline | Status |
|---|---|---|
| mne / mne-bids | Data I/O, preprocessing, PSD, connectivity | **Primary** |
| antropy | Nonlinear feature extraction | **Primary** |
| EEGExtract | MFCC, ARMA, coherence, Granger | **Primary (manual install)** |
| invertmeeg | Source-level S1 projection | **Adopted for source stage** |
| eegpipe-python | Epoch QC, topographic plots | **Supporting** |
| best-toolbox | Classification utilities, normalization | **Supporting** |
| python-neo | Format bridge (optional) | **Optional** |
| eeg-expy | Online delivery/basic offline | **Not adopted** |
| eegprep (sccn) | Robust re-reference | **Optional (no PyPI release)** |

Proceed to **Notebook 02** for data loading and preprocessing.